# SPMamba — Train trên Colab + theo dõi từ xa qua Weights & Biases

Notebook này train SPMamba và **đẩy log (train_loss theo từng step, val_loss, lr) lên W&B**
để bạn theo dõi từ xa ở wandb.ai — kể cả khi đóng máy.

**Chuẩn bị 1 lần:** thêm secret `WANDB_API_KEY` vào Colab Secrets (biểu tượng 🔑 ở thanh trái).
Lấy key tại https://wandb.ai/authorize

Chạy **lần lượt** từng cell từ trên xuống. Cell cuối (train) sẽ in **URL của W&B run** —
mở URL đó để xem biểu đồ loss cập nhật live.

In [ ]:
# === Cell 1: Cấu hình đường dẫn + W&B ===
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# ----- ĐƯỜNG DẪN (chỉnh nếu cần) -----
REPO_URL = "https://github.com/hoanganh0106/spmamba.git"
REPO_DIR = "/content/spmamba"
ENV_DIR  = "/content/spmamba-env"
MM       = "/content/bin/micromamba"

DRIVE_H5_PATH = "/content/drive/MyDrive/data_30h.h5"   # dataset gốc trên Drive
LOCAL_H5_PATH = "/content/data.h5"                      # copy về SSD local cho nhanh
CONFIG_PATH   = f"{REPO_DIR}/configs/spmamba-3speakers.yml"   # config Colab T4
CKPT_DIR      = "/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers"

# ----- WEIGHTS & BIASES -----
WANDB_PROJECT  = "spmamba"
WANDB_RUN_NAME = "SPMamba-3Speakers-T4"

# Lấy API key: ưu tiên Colab Secrets -> biến môi trường -> dán tay
WANDB_API_KEY = ""
try:
    from google.colab import userdata
    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
except Exception:
    pass
WANDB_API_KEY = WANDB_API_KEY or os.environ.get("WANDB_API_KEY", "")
# WANDB_API_KEY = "dan-key-vao-day-neu-khong-dung-Secrets"

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("[wandb] Da co WANDB_API_KEY OK")
else:
    print("[wandb] CHUA co WANDB_API_KEY! Them secret ten 'WANDB_API_KEY' vao Colab Secrets (icon khoa) roi chay lai cell nay.")

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
print("[paths]")
for k in ["REPO_DIR", "ENV_DIR", "DRIVE_H5_PATH", "LOCAL_H5_PATH", "CONFIG_PATH", "CKPT_DIR"]:
    print(f"  {k} = {globals()[k]}")

In [ ]:
# === Cell 2: Cài môi trường + thư viện + đăng nhập W&B ===
import os, subprocess, shutil
from pathlib import Path

os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba-root"
os.environ["PATH"] = "/content/bin:" + os.environ.get("PATH", "")

INSTALL_DEPS       = True    # doi thanh False sau khi env da cai xong de chay lai nhanh hon
FORCE_RECREATE_ENV = False   # chi bat True khi env hong nang

def run(cmd, cwd=None, check=True):
    print(f"\n[cmd] {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, cwd=cwd, env=os.environ.copy(), text=True, check=check)

Path('/content/bin').mkdir(parents=True, exist_ok=True)
if not Path(MM).exists():
    run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba")
run(f"{MM} --version")

if FORCE_RECREATE_ENV and Path(ENV_DIR).exists():
    shutil.rmtree(ENV_DIR)
if not Path(ENV_DIR).exists():
    run(f"{MM} create -p {ENV_DIR} python=3.10 --yes -q")

if INSTALL_DEPS:
    run(f"{MM} run -p {ENV_DIR} python -m pip install -U pip -q")
    run(f"{MM} run -p {ENV_DIR} pip install 'setuptools<70' typeguard==2.13.3 h5py pytorch-lightning==2.0.2 einops soundfile librosa fast-bss-eval speechbrain==0.5.14 hydra-core rich torch-complex packaging wandb -q")
    run(f"{MM} run -p {ENV_DIR} pip install --force-reinstall scipy -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch-mir-eval torch-optimizer transformers -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch==2.5.0 torchaudio==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall -q")
    run(f"{MM} run -p {ENV_DIR} pip install 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/causal_conv1d-1.6.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' 'https://github.com/state-spaces/mamba/releases/download/v2.3.0/mamba_ssm-2.3.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' --no-deps --force-reinstall -q")

# Kiem tra nhanh
run(f"{MM} run -p {ENV_DIR} python -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())\"")
run(f"{MM} run -p {ENV_DIR} python -c \"from mamba_ssm.modules.mamba_simple import Mamba; print('mamba_ssm OK')\"")
run(f"{MM} run -p {ENV_DIR} python -c \"import wandb; print('wandb', wandb.__version__)\"")
# Dang nhap wandb (dung WANDB_API_KEY da set o Cell 1)
run(f"{MM} run -p {ENV_DIR} python -c \"import os, wandb; k=os.environ.get('WANDB_API_KEY'); print('wandb login:', wandb.login(key=k, relogin=True)) if k else print('CHUA co WANDB_API_KEY - chay lai Cell 1')\"")
run("nvidia-smi", check=False)

In [ ]:
# === Cell 3: Clone/cap nhat repo ===
from pathlib import Path
import os

if not Path(REPO_DIR).exists():
    run(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print(f"[OK] Repo da co: {REPO_DIR}")
    run("git pull --ff-only", cwd=REPO_DIR, check=False)

os.chdir(REPO_DIR)
print("[cwd]", os.getcwd())
run("git rev-parse --short HEAD", cwd=REPO_DIR, check=False)
for f in ["audio_train.py", "look2hear/datas/h5_datamodule.py", CONFIG_PATH]:
    print(f"[check] {f} =", Path(f).exists())

In [ ]:
# === Cell 4: Copy dataset ve SSD local ===
from pathlib import Path
import shutil, time

src, dst = Path(DRIVE_H5_PATH), Path(LOCAL_H5_PATH)
gb = lambda n: n / (1024 ** 3)
print("[dataset] Drive:", src, "exists =", src.exists())
if not src.exists():
    raise FileNotFoundError(f"Khong tim thay dataset tren Drive: {src}")
print(f"[dataset] Drive size: {gb(src.stat().st_size):.2f} GiB")

if dst.exists() and dst.stat().st_size == src.stat().st_size:
    print(f"[OK] Local H5 da co, cung size: {dst}")
else:
    print(f"[copy] Copy ve SSD local: {src} -> {dst} (mat vai phut)...", flush=True)
    t0 = time.time()
    shutil.copy2(src, dst)
    print(f"[OK] Copy xong sau {(time.time()-t0)/60:.1f} phut, size={gb(dst.stat().st_size):.2f} GiB")

In [ ]:
# === Cell 5: TRAIN + theo doi tu xa qua W&B ===
# Cell nay stream output ra notebook, dong thoi in [heartbeat] moi 30s (GPU util + im_lang)
# de ban biet no DANG CHAY ngay ca khi chua co progress bar.
# Theo doi bieu do loss live tai URL wandb.ai in ben duoi.
import os, subprocess, select, time, shutil
from pathlib import Path

os.chdir(REPO_DIR)

# --- Resume bridge: neu chua co last.ckpt nhung co epoch=*-best.ckpt -> tao last.ckpt tu cai moi nhat ---
ckpt_root = Path(CKPT_DIR); ckpt_root.mkdir(parents=True, exist_ok=True)
last_path = ckpt_root / "last.ckpt"
if not last_path.exists():
    others = [p for p in ckpt_root.glob("*.ckpt") if not p.name.startswith("last")]
    if others:
        newest = max(others, key=lambda p: p.stat().st_mtime)
        shutil.copy2(newest, last_path)
        print("[resume] tao last.ckpt tu", newest.name)
resume = last_path.exists()

cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "audio_train.py",
       "--conf_dir", CONFIG_PATH, "--checkpoint_dir", CKPT_DIR,
       "--wandb", "--wandb_project", WANDB_PROJECT, "--wandb_run_name", WANDB_RUN_NAME]
if resume:
    cmd.append("--resume")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = REPO_DIR
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def gpu_line():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True, timeout=5).strip()
    except Exception as e:
        return f"n/a ({e!r})"

print("[train] lenh:", " ".join(cmd))
print("[train] resume:", resume)
print("[train] >>> Mo wandb.ai de theo doi tu xa. URL hien ben duoi khi W&B khoi tao. <<<\n", flush=True)

proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
start = last_out = time.time(); last_hb = 0.0
try:
    while True:
        ready, _, _ = select.select([proc.stdout], [], [], 1.0)
        if ready:
            line = proc.stdout.readline()
            if line:
                last_out = time.time()
                print(line, end="", flush=True)
        now = time.time()
        if now - last_hb >= 30:
            print(f"[heartbeat] chay={(now-start)/60:.1f}min | im_lang={now-last_out:.0f}s | GPU(util%,used,total MiB)={gpu_line()}", flush=True)
            last_hb = now
        if proc.poll() is not None:
            for rest in proc.stdout:
                print(rest, end="", flush=True)
            print(f"\n[train] ket thuc - exit code {proc.returncode}")
            break
except KeyboardInterrupt:
    proc.terminate()
    print("\n[train] da dung (KeyboardInterrupt)")

In [ ]:
# === Cell 6: Liet ke checkpoint da luu (chay luc nao cung duoc) ===
from pathlib import Path
import time

p = Path(CKPT_DIR)
print("[checkpoints]", p)
ckpts = sorted(p.glob("*.ckpt"), key=lambda x: x.stat().st_mtime, reverse=True)
for ck in ckpts[:15]:
    print(f"  {ck.name:32s} {ck.stat().st_size/1e6:6.1f} MB  {time.ctime(ck.stat().st_mtime)}")
bj = p / "best_k_models.json"
if bj.exists():
    print("\nbest_k_models.json:\n", bj.read_text(encoding="utf-8"))